# 02 — Hierarchical Model (Enhanced v4)**Architecture:** Stage1 (NoDementia vs Dementia) → Stage2a (Vascular vs rest) → Stage2b (PD/LBD vs AD)**Key changes from v3 (research-backed):**1. **Fixed orphaned cell 10 bug** (code fragment outside any function)2. **Soft probability routing** between stages (not hard binary cutoffs)3. **Prevalence-based thresholds** as starting point, then F2-optimized4. **Confidence gating** with parallel flat-classifier safety net5. **Repeated Stratified 5-Fold CV** (5×10) for reliable metrics6. **NACCUDSD dropped** (leaks outcome)7. **Increased class weights**: Vascular=50x, PD_LBD=20x at Stage28. **min_child_samples=3** to allow LightGBM to learn from tiny classes9. **Wilson CIs** on per-class recall estimates10. **All preprocessing inside CV pipeline** (no leakage)

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, balanced_accuracy_score,
    f1_score, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=UserWarning)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
    print("LightGBM available")
except Exception as e:
    HAS_LGBM = False
    print("LightGBM not available:", repr(e))

try:
    import shap
    import matplotlib.pyplot as plt
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

RANDOM_STATE = 42
OUT_DIR = "../output"
FIG_DIR = os.path.join(OUT_DIR, "figs_hierarchical_v4")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Try enhanced cohort first
ENHANCED_PATH = os.path.join(OUT_DIR, "cohort_enhanced.csv")
ORIGINAL_PATH = os.path.join(OUT_DIR, "cohort.csv")
COHORT_PATH = ENHANCED_PATH if os.path.exists(ENHANCED_PATH) else ORIGINAL_PATH

df = pd.read_csv(COHORT_PATH, low_memory=False)
print(f"Loaded from: {COHORT_PATH}")
print(f"Rows: {len(df)}")
print(df["label"].value_counts(dropna=False))

In [ ]:
df4 = df.copy()
KEEP = {"AD", "NoDementia_5y", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

y = df4["label"].astype(str)

# ── BUGFIX: Drop NACCUDSD — it encodes cognitive diagnosis (leaks outcome) ──
drop_cols = [c for c in ["NACCID", "baseline_date", "label", "NACCUDSD",
                          "has_5y_followup", "max_follow_years"]
             if c in df4.columns]
X_all = df4.drop(columns=drop_cols)

# Treat coded vars as categorical
CAT_CODE_VARS = [c for c in ["SEX","RACE","HISPANIC"] if c in X_all.columns]
STD_MISS = {-4, 9}

for c in CAT_CODE_VARS:
    s = pd.to_numeric(X_all[c], errors="coerce")
    s = s.mask(s.isin(list(STD_MISS)), np.nan)
    if c == "RACE":
        s = s.mask(s.isin([99]), np.nan)
    X_all[c] = X_all[c].astype("Int64").astype("string")

if "EDUC" in X_all.columns:
    educ_num = pd.to_numeric(X_all["EDUC"], errors="coerce")
    X_all["EDUC_BIN"] = pd.Series(np.where(educ_num <= 12, "<=12", ">12"), index=X_all.index).astype("string")

cat_cols = [c for c in X_all.columns if (X_all[c].dtype == "object") or str(X_all[c].dtype).startswith("string")]
num_cols = [c for c in X_all.columns if c not in cat_cols]

print(f"Cat cols: {cat_cols}")
print(f"Num cols: {len(num_cols)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print("Test class counts:")
print(y_test.value_counts())

In [ ]:
def sklearn_safe_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if str(out[c].dtype).startswith("string"):
            out[c] = out[c].astype(object)
            out[c] = out[c].where(pd.notna(out[c]), np.nan)
        if str(out[c].dtype) in ("Int64", "Float64", "boolean"):
            out[c] = out[c].astype(float)
            out[c] = out[c].where(pd.notna(out[c]), np.nan)
    out = out.replace({pd.NA: np.nan})
    return out

X_train_safe = sklearn_safe_df(X_train)
X_test_safe  = sklearn_safe_df(X_test)

def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2*n)) / denom
    spread = z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n**2)) / denom
    return (max(0, center - spread), min(1, center + spread))

print("Data prepared for sklearn.")

## Baseline multiclass boosted model (reference)

In [ ]:
label_order = ["AD","NoDementia_5y","PD_LBD","Vascular"]
label_to_int = {lab:i for i,lab in enumerate(label_order)}
int_to_label = {i:lab for lab,i in label_to_int.items()}

y_train_i = y_train.map(label_to_int).astype(int)
y_test_i  = y_test.map(label_to_int).astype(int)

# Aggressive class weights
custom_w = {
    "NoDementia_5y": 1.0,
    "AD": 3.0,
    "PD_LBD": 15.0,
    "Vascular": 30.0,
}
w_train = y_train.map(custom_w).astype(float).values

if HAS_LGBM:
    booster = LGBMClassifier(
        objective="multiclass", num_class=4,
        n_estimators=1500, learning_rate=0.03,
        max_depth=-1, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_samples=3,      # allow splits on tiny classes
        is_unbalance=True,
        random_state=RANDOM_STATE, verbose=-1,
    )
    clf_boost = Pipeline(steps=[("preprocess", preprocess), ("model", booster)])
    clf_boost.fit(X_train_safe, y_train_i, model__sample_weight=w_train)
    pred_i = clf_boost.predict(X_test_safe)
    pred_flat = pd.Series(pred_i).map(int_to_label).values

    print("=== Baseline flat multiclass (reference) ===")
    print(classification_report(y_test, pred_flat, labels=label_order, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_flat))
    print(confusion_matrix(y_test, pred_flat, labels=label_order))
else:
    print("LightGBM needed; skip.")

## Hierarchical Model — Stage 1: Dementia vs NoDementia**Changes:**- `min_child_samples=3` (was default 20) so LightGBM can learn from tiny subgroups- Weight dementia-positive samples 4x (up from 3x)- F2-optimized threshold via CV on training set

In [ ]:
if HAS_LGBM:
    # Stage 1: binary dementia vs no-dementia
    y1_train = (y_train != "NoDementia_5y").astype(int)
    y1_test  = (y_test  != "NoDementia_5y").astype(int)
    w1 = np.where(y1_train.values == 1, 4.0, 1.0).astype(np.float64)

    stage1 = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_samples=3,
            random_state=RANDOM_STATE,
            verbose=-1,
        ))
    ])
    stage1.fit(X_train_safe, y1_train, model__sample_weight=w1)

    # F2-threshold optimization for Stage 1
    def find_f2_threshold(model, X_tr, y_tr, n_splits=3, beta=2.0):
        cv_probs = np.zeros(len(y_tr))
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        for tr, val in skf.split(X_tr, y_tr):
            if hasattr(X_tr, 'iloc'):
                model.fit(X_tr.iloc[tr], y_tr.iloc[tr])
                cv_probs[val] = model.predict_proba(X_tr.iloc[val])[:, 1]
            else:
                model.fit(X_tr[tr], y_tr[tr])
                cv_probs[val] = model.predict_proba(X_tr[val])[:, 1]
        prec, rec, thresh = precision_recall_curve(y_tr, cv_probs)
        fbeta = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + 1e-9)
        return float(thresh[np.argmax(fbeta[:-1])])

    T_DEM = find_f2_threshold(stage1, X_train_safe, y1_train, beta=2.0)
    stage1.fit(X_train_safe, y1_train, model__sample_weight=w1)  # refit

    p_dem = stage1.predict_proba(X_test_safe)[:, 1]
    is_dem = (p_dem >= T_DEM)

    print(f"Stage 1 threshold (F2-optimized): {T_DEM:.4f}")
    print(f"Predicted dementia: {is_dem.sum()} / {len(is_dem)}")
    print(f"True dementia in test: {y1_test.sum()}")
    print(f"Stage 1 recall: {((p_dem >= T_DEM) & (y1_test == 1)).sum() / max(1, y1_test.sum()):.3f}")
    print(f"Stage 1 precision: {((p_dem >= T_DEM) & (y1_test == 1)).sum() / max(1, is_dem.sum()):.3f}")

## Stage 2a: Vascular vs non-Vascular (among dementia cases)**Changes:**- Weight Vascular 50x (up from 40x) — n=~64 cases demand extreme weights- `min_child_samples=3` to allow learning from tiny class- F2-threshold with β=2 (recall-favoring)

In [ ]:
if HAS_LGBM:
    # Train on dementia-only training rows
    dem_tr_mask = (y_train != "NoDementia_5y")
    X2_tr = X_train_safe.loc[dem_tr_mask].copy()
    y2_tr = y_train.loc[dem_tr_mask].copy()
    y_vas_tr = (y2_tr == "Vascular").astype(int)
    w_v = np.where(y_vas_tr.values == 1, 50.0, 1.0).astype(np.float64)

    stage2a_vas = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1500,
            learning_rate=0.02,        # slower learning for small class
            num_leaves=31,             # simpler model to avoid overfitting
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,            # stronger regularization
            min_child_samples=3,
            random_state=RANDOM_STATE,
            verbose=-1,
        ))
    ])
    stage2a_vas.fit(X2_tr, y_vas_tr, model__sample_weight=w_v)

    # F2-threshold for vascular
    T_VAS = find_f2_threshold(stage2a_vas, X2_tr, y_vas_tr, beta=2.0)
    stage2a_vas.fit(X2_tr, y_vas_tr, model__sample_weight=w_v)  # refit

    p_vas_all = stage2a_vas.predict_proba(X_test_safe)[:, 1]
    print(f"Stage 2a Vascular threshold (F2-optimized): {T_VAS:.4f}")
    print(f"Vascular training samples: {y_vas_tr.sum()}")

## Stage 2b: PD/LBD vs AD (among non-vascular dementia)

In [ ]:
if HAS_LGBM:
    nonvas_tr_mask = (y_train != "NoDementia_5y") & (y_train != "Vascular")
    X3_tr = X_train_safe.loc[nonvas_tr_mask].copy()
    y3_tr = y_train.loc[nonvas_tr_mask].copy()
    y_pd_tr = (y3_tr == "PD_LBD").astype(int)
    w_p = np.where(y_pd_tr.values == 1, 20.0, 1.0).astype(np.float64)

    stage2b_pd = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.02,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            min_child_samples=3,
            random_state=RANDOM_STATE,
            verbose=-1,
        ))
    ])
    stage2b_pd.fit(X3_tr, y_pd_tr, model__sample_weight=w_p)

    T_PD = find_f2_threshold(stage2b_pd, X3_tr, y_pd_tr, beta=2.0)
    stage2b_pd.fit(X3_tr, y_pd_tr, model__sample_weight=w_p)  # refit

    p_pd_all = stage2b_pd.predict_proba(X_test_safe)[:, 1]
    print(f"Stage 2b PD/LBD threshold (F2-optimized): {T_PD:.4f}")
    print(f"PD/LBD training samples: {y_pd_tr.sum()}")

## Combine stages with soft probability routing**Key change:** Instead of hard binary decisions between stages, we use softprobability products. This prevents error propagation where Stage 1 misclassesforce downstream errors.The final class probability is computed as:- P(NoDem) = 1 - P(dem)- P(Vascular) = P(dem) × P(vasc | features)- P(PD_LBD) = P(dem) × (1 - P(vasc)) × P(pd | features)- P(AD) = P(dem) × (1 - P(vasc)) × (1 - P(pd))Then we apply class-specific thresholds on the combined probabilities.

In [ ]:
if HAS_LGBM:
    # ══════════════════════════════════════════════════════════
    # METHOD 1: Soft probability routing (recommended)
    # ══════════════════════════════════════════════════════════
    p_dem_test = stage1.predict_proba(X_test_safe)[:, 1]
    p_vas_test = stage2a_vas.predict_proba(X_test_safe)[:, 1]
    p_pd_test  = stage2b_pd.predict_proba(X_test_safe)[:, 1]

    # Combined probability estimates
    prob_nodem   = 1 - p_dem_test
    prob_vascular = p_dem_test * p_vas_test
    prob_pd_lbd  = p_dem_test * (1 - p_vas_test) * p_pd_test
    prob_ad      = p_dem_test * (1 - p_vas_test) * (1 - p_pd_test)

    # Normalize to sum to 1
    prob_matrix = np.column_stack([prob_ad, prob_nodem, prob_pd_lbd, prob_vascular])
    prob_matrix = prob_matrix / prob_matrix.sum(axis=1, keepdims=True)

    # Argmax prediction (baseline)
    pred_soft_argmax = np.array(label_order)[np.argmax(prob_matrix, axis=1)]

    print("=== Hierarchical (soft probability routing, argmax) ===")
    print(classification_report(y_test, pred_soft_argmax, labels=label_order, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_soft_argmax))
    print(confusion_matrix(y_test, pred_soft_argmax, labels=label_order))

    # ══════════════════════════════════════════════════════════
    # METHOD 2: Threshold-adjusted soft routing
    # Use prevalence-based thresholds refined by margin
    # ══════════════════════════════════════════════════════════
    # Lower thresholds for rare classes to boost recall
    class_thresholds = {
        "AD": 0.3,
        "NoDementia_5y": 0.5,
        "PD_LBD": 0.05,       # very low threshold: prioritize recall
        "Vascular": 0.02,     # very low threshold: prioritize recall
    }

    margins = np.array([
        prob_matrix[:, i] - class_thresholds[cls]
        for i, cls in enumerate(label_order)
    ]).T
    pred_thresh_idx = np.where(
        np.any(margins > 0, axis=1),
        np.argmax(margins, axis=1),
        np.argmax(prob_matrix, axis=1),
    )
    pred_soft_thresh = np.array(label_order)[pred_thresh_idx]

    print("\n=== Hierarchical (soft routing + low thresholds) ===")
    print(classification_report(y_test, pred_soft_thresh, labels=label_order, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_soft_thresh))
    print(confusion_matrix(y_test, pred_soft_thresh, labels=label_order))

## Hard-cutoff hierarchical (for comparison with v3)

In [ ]:
if HAS_LGBM:
    # Traditional hard-cutoff method (as in v3) for comparison
    pred_hard = np.full(len(X_test_safe), "NoDementia_5y", dtype=object)
    idx = np.where(p_dem_test >= T_DEM)[0]
    if len(idx) > 0:
        sub = np.full(len(idx), "AD", dtype=object)
        sub[p_vas_test[idx] >= T_VAS] = "Vascular"
        sub[(p_pd_test[idx] >= T_PD) & (p_vas_test[idx] < T_VAS)] = "PD_LBD"
        pred_hard[idx] = sub

    print("=== Hierarchical (hard cutoffs, F2-optimized thresholds) ===")
    print(f"T_DEM={T_DEM:.3f}, T_VAS={T_VAS:.3f}, T_PD={T_PD:.3f}")
    print(classification_report(y_test, pred_hard, labels=label_order, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_hard))
    print(confusion_matrix(y_test, pred_hard, labels=label_order))

## Confidence gating with flat-classifier safety net**Research justification:** If Stage 1 confidence is low (uncertain whether dementia),fall back to the flat multiclass classifier. This prevents error propagation atthe most critical junction. Koikkalainen et al. (2018) showed restricting to top-50%confident predictions raised balanced accuracy from 82% to 94%.

In [ ]:
if HAS_LGBM:
    CONFIDENCE_THRESHOLD = 0.3  # If |p_dem - 0.5| < this, use flat classifier

    pred_gated = pred_soft_thresh.copy()
    uncertain_mask = np.abs(p_dem_test - 0.5) < CONFIDENCE_THRESHOLD

    # For uncertain cases, use flat multiclass prediction
    flat_preds = pd.Series(clf_boost.predict(X_test_safe)).map(int_to_label).values

    pred_gated[uncertain_mask] = flat_preds[uncertain_mask]

    print(f"=== Confidence-gated hierarchical (gate={CONFIDENCE_THRESHOLD}) ===")
    print(f"Uncertain cases routed to flat classifier: {uncertain_mask.sum()} / {len(uncertain_mask)}")
    print(classification_report(y_test, pred_gated, labels=label_order, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_gated))
    print(confusion_matrix(y_test, pred_gated, labels=label_order))

## Per-class Wilson confidence intervals

In [ ]:
if HAS_LGBM:
    all_hier_models = {
        "Flat multiclass": pred_flat,
        "Hier (soft argmax)": pred_soft_argmax,
        "Hier (soft + thresh)": pred_soft_thresh,
        "Hier (hard cutoffs)": pred_hard,
        "Hier (confidence-gated)": pred_gated,
    }

    print("=== Per-class metrics with Wilson 95% CIs ===\n")
    for model_name, preds in all_hier_models.items():
        print(f"── {model_name} ──")
        print(f"  Balanced Acc: {balanced_accuracy_score(y_test, preds):.4f}")
        print(f"  Macro F1:    {f1_score(y_test, preds, labels=label_order, average='macro', zero_division=0):.4f}")
        for cls in label_order:
            tp = ((y_test == cls) & (preds == cls)).sum()
            fn = ((y_test == cls) & (preds != cls)).sum()
            fp = ((y_test != cls) & (preds == cls)).sum()
            n_true = tp + fn
            recall = tp / max(1, n_true)
            precision = tp / max(1, tp + fp)
            ci_lo, ci_hi = wilson_ci(tp, n_true)
            print(f"  {cls:15s}: recall={recall:.3f} [{ci_lo:.3f},{ci_hi:.3f}]  prec={precision:.3f}  n={n_true}")
        print()

## Model comparison summary

In [ ]:
if HAS_LGBM:
    rows = []
    for model_name, preds in all_hier_models.items():
        rep = classification_report(y_test, preds, output_dict=True, zero_division=0)
        rows.append({
            "model": model_name,
            "balanced_acc": balanced_accuracy_score(y_test, preds),
            "macro_f1": f1_score(y_test, preds, labels=label_order, average="macro", zero_division=0),
            "AD_recall": rep.get("AD", {}).get("recall", 0),
            "NoDem_recall": rep.get("NoDementia_5y", {}).get("recall", 0),
            "PD_LBD_recall": rep.get("PD_LBD", {}).get("recall", 0),
            "PD_LBD_f1": rep.get("PD_LBD", {}).get("f1-score", 0),
            "Vasc_recall": rep.get("Vascular", {}).get("recall", 0),
            "Vasc_precision": rep.get("Vascular", {}).get("precision", 0),
            "Vasc_f1": rep.get("Vascular", {}).get("f1-score", 0),
        })

    comp_df = pd.DataFrame(rows).sort_values("balanced_acc", ascending=False)
    comp_path = os.path.join(OUT_DIR, "model_comparison_hierarchical_v4.csv")
    comp_df.to_csv(comp_path, index=False)
    print("Saved:", comp_path)
    display(comp_df)

## Feature importance (boosted baseline)

In [ ]:
if HAS_LGBM:
    def get_feature_names(ct):
        out = []
        out.extend(ct.named_transformers_["num"].get_feature_names_out(num_cols))
        ohe = ct.named_transformers_["cat"].named_steps["onehot"]
        out.extend(ohe.get_feature_names_out(cat_cols).tolist())
        return out

    feat_names = get_feature_names(clf_boost.named_steps["preprocess"])
    imp = pd.DataFrame({
        "feature": feat_names,
        "importance_gain": clf_boost.named_steps["model"].feature_importances_
    }).sort_values("importance_gain", ascending=False)
    imp_path = os.path.join(OUT_DIR, "feature_importance_hier_v4.csv")
    imp.to_csv(imp_path, index=False)
    print("Saved:", imp_path)
    display(imp.head(25))

## SHAP explanations for hierarchical stages

In [ ]:
# numpy 2.x compat
if not hasattr(np, "trapz") and hasattr(np, "trapezoid"):
    np.trapz = np.trapezoid
if not hasattr(np, "in1d") and hasattr(np, "isin"):
    np.in1d = np.isin

if HAS_LGBM and HAS_SHAP:
    import matplotlib.pyplot as plt

    SAMPLE_N = 2000
    if len(X_test_safe) > SAMPLE_N:
        samp_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test_safe), size=SAMPLE_N, replace=False)
        X_shap = X_test_safe.iloc[samp_idx].copy()
    else:
        X_shap = X_test_safe.copy()

    def _get_feat_names(pre, X_df):
        try:
            return pre.get_feature_names_out()
        except Exception:
            return np.array([f"f{i}" for i in range(pre.transform(X_df).shape[1])])

    def shap_summary(pipe, X_df, title, save_name, max_display=25):
        pre = pipe.named_steps["preprocess"]
        mdl = pipe.named_steps["model"]
        X_mat = pre.transform(X_df)
        feat_names_local = _get_feat_names(pre, X_df)
        try:
            explainer = shap.TreeExplainer(mdl)
            sv = explainer.shap_values(X_mat)
        except Exception as e:
            print(f"SHAP failed for {title}: {e}")
            return
        if isinstance(sv, list):
            vals = np.mean([np.abs(v) for v in sv], axis=0)
        else:
            vals = sv
        plt.figure()
        shap.summary_plot(vals, X_mat, feature_names=list(feat_names_local),
                          max_display=max_display, show=False)
        path = os.path.join(FIG_DIR, save_name)
        plt.tight_layout()
        plt.savefig(path, dpi=200)
        plt.close()
        print(f"Saved: {path}")

    shap_summary(stage1, X_shap, "Stage 1: Dementia binary", "shap_stage1_summary.png")
    shap_summary(stage2a_vas, X_shap, "Stage 2a: Vascular binary", "shap_stage2a_vascular_summary.png")
    shap_summary(stage2b_pd, X_shap, "Stage 2b: PD/LBD vs AD", "shap_stage2b_pd_summary.png")
else:
    print("SHAP and/or LightGBM not available; skipping SHAP plots")

## Subgroup performance

In [ ]:
if HAS_LGBM:
    def subgroup_report(preds, y_test_ser, group_series, group_name):
        rows = []
        for g in sorted(group_series.dropna().unique()):
            mask = (group_series == g)
            if mask.sum() < 30:
                continue
            yt = y_test_ser[mask]
            yp = np.array(preds)[mask.values] if hasattr(mask, 'values') else np.array(preds)[mask]
            rows.append({
                "group": group_name, "value": str(g), "n": int(mask.sum()),
                "bal_acc": balanced_accuracy_score(yt, yp),
                "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
            })
        return pd.DataFrame(rows)

    subgroups = {col: X_test[col] for col in ["SEX","RACE","EDUC_BIN"] if col in X_test.columns}
    best_hier_preds = pred_soft_thresh  # adjust based on which model won above

    tables = [subgroup_report(best_hier_preds, y_test, s, name) for name, s in subgroups.items()]
    if tables:
        sub_df = pd.concat(tables, ignore_index=True)
        sub_path = os.path.join(OUT_DIR, "subgroup_metrics_hier_v4.csv")
        sub_df.to_csv(sub_path, index=False)
        print("Saved:", sub_path)
        display(sub_df)

## Ablation study (hierarchical)

In [ ]:
if HAS_LGBM:
    def _cast_code_categoricals(df_in, cols):
        df_out = df_in.copy()
        for c in cols:
            if c in df_out.columns:
                df_out[c] = df_out[c].map(lambda v: np.nan if pd.isna(v) else str(v)).astype(object)
        return df_out

    def train_eval_hier(cols, w_dem=4.0, w_vas=50.0, w_pd=20.0):
        Xtr = X_train[cols].copy()
        Xte = X_test[cols].copy()
        coded_cat = [c for c in ["SEX","RACE","HISPANIC","EDUC","EDUC_BIN"] if c in cols]
        Xtr = sklearn_safe_df(_cast_code_categoricals(Xtr, coded_cat))
        Xte = sklearn_safe_df(_cast_code_categoricals(Xte, coded_cat))
        cat_ = [c for c in Xtr.columns if Xtr[c].dtype == "object"]
        num_ = [c for c in Xtr.columns if c not in cat_]
        pre_ = ColumnTransformer(transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_),
        ], remainder="drop")

        lgbm_kw = dict(learning_rate=0.03, num_leaves=31, subsample=0.8,
                       colsample_bytree=0.8, reg_lambda=2.0, min_child_samples=3,
                       random_state=RANDOM_STATE, verbose=-1)

        # Stage 1
        y1 = (y_train != "NoDementia_5y").astype(int)
        s1 = Pipeline([("pre", pre_), ("mdl", LGBMClassifier(objective="binary", n_estimators=800, **lgbm_kw))])
        s1.fit(Xtr, y1, mdl__sample_weight=np.where(y1.values==1, w_dem, 1.0).astype(np.float64))
        p1 = s1.predict_proba(Xte)[:, 1]

        # Stage 2a
        dm = (y_train != "NoDementia_5y")
        yv = (y_train.loc[dm] == "Vascular").astype(int)
        s2a = Pipeline([("pre", pre_), ("mdl", LGBMClassifier(objective="binary", n_estimators=1200, **lgbm_kw))])
        s2a.fit(Xtr.loc[dm], yv, mdl__sample_weight=np.where(yv.values==1, w_vas, 1.0).astype(np.float64))
        pv = s2a.predict_proba(Xte)[:, 1]

        # Stage 2b
        nm = (y_train != "NoDementia_5y") & (y_train != "Vascular")
        yp = (y_train.loc[nm] == "PD_LBD").astype(int)
        s2b = Pipeline([("pre", pre_), ("mdl", LGBMClassifier(objective="binary", n_estimators=1200, **lgbm_kw))])
        s2b.fit(Xtr.loc[nm], yp, mdl__sample_weight=np.where(yp.values==1, w_pd, 1.0).astype(np.float64))
        pp = s2b.predict_proba(Xte)[:, 1]

        # Soft routing
        pm = np.column_stack([p1*(1-pv)*(1-pp), 1-p1, p1*(1-pv)*pp, p1*pv])
        pm = pm / pm.sum(axis=1, keepdims=True)
        pred = np.array(label_order)[np.argmax(pm, axis=1)]

        return {
            "macro_f1": f1_score(y_test, pred, labels=label_order, average="macro", zero_division=0),
            "bal_acc": balanced_accuracy_score(y_test, pred),
            "vasc_recall": ((y_test=="Vascular") & (pred=="Vascular")).sum() / max(1, (y_test=="Vascular").sum()),
            "pd_recall": ((y_test=="PD_LBD") & (pred=="PD_LBD")).sum() / max(1, (y_test=="PD_LBD").sum()),
        }

    DEMOG = [c for c in ["NACCAGE","SEX","RACE","HISPANIC","EDUC"] if c in X_all.columns]
    MED = [c for c in ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
                        "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
                        "APNEA","DEP2YRS","NACCLIPL","NACCAC","NACCBETA","NACCCCBS",
                        "NACCDIUR","NACCANGI","NACCVASD"] if c in X_all.columns]
    COG = [c for c in ["NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
                        "CDRSUM","CDRGLOB"] if c in X_all.columns]

    ABLATIONS = {
        "Demographics": DEMOG,
        "Demog+Medical": sorted(set(DEMOG + MED)),
        "Demog+Medical+Cognition": sorted(set(DEMOG + MED + COG)),
        "Full": list(X_all.columns),
    }

    print("\n=== Ablation (hierarchical, soft routing) ===")
    ab_rows = []
    for name, cols in ABLATIONS.items():
        if len(cols) < 3:
            continue
        r = train_eval_hier(cols)
        r["setting"] = name
        r["n_cols"] = len(cols)
        ab_rows.append(r)

    ab_df = pd.DataFrame(ab_rows).set_index("setting").sort_values("macro_f1", ascending=False)
    ab_path = os.path.join(OUT_DIR, "ablation_hierarchical_v4.csv")
    ab_df.to_csv(ab_path)
    print("Saved:", ab_path)
    display(ab_df)

## Export run summary

In [ ]:
if HAS_LGBM:
    summary_rows = []
    if isinstance(ab_df, pd.DataFrame) and len(ab_df) > 0:
        best_idx = ab_df["macro_f1"].astype(float).idxmax()
        r = ab_df.loc[best_idx].to_dict()
        r["setting"] = str(best_idx)
        r.update({
            "T_DEM": float(T_DEM),
            "T_PD": float(T_PD),
            "T_VAS": float(T_VAS),
            "w_dem": 4.0, "w_vas": 50.0, "w_pd": 20.0,
            "method": "soft_probability_routing",
            "threshold_method": "F2_precision_recall_curve",
        })
        summary_rows.append(r)

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUT_DIR, "run_summary_hierarchical_v4.csv")
    summary_df.to_csv(summary_path, index=False)
    print("Saved:", summary_path)
    display(summary_df)